<table>
  <tr>
    <td><div align="left"><font size="30">Ball-beam system</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

**Modeling the classic ball and beam system.**

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package (Colab, JupyterLite or local), generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}...')
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(extras=['control'], min_version='1.5.0')


# Theory

A ball of mass $m$ rolls in a groove on a beam.  The beam is pivoted at its centre and has an angle $\theta$ with respect
to horizontal.  The position of the ball is measured with respect to the pivot point.
The beam angle $\theta$ is actuated in order to control the
ball position.
The ball experiences viscous friction $b$.

By applying the Euler-Lagrange method the acceleration of the ball along the beam is given by
$$
\left(m+\frac{J}{R^{2}}\right) \ddot{x} + mg\sin \theta -mx\dot{\theta }^{2}=0
$$
For a sphere with uniform mass distribution $J = \frac{2}{5}mR^2$ so the coefficient of $\ddot{x}$
becomes $\frac{5}{7}$ 
This fraction represents the partitioning of energy between linear and rotational movement. Because the ball must roll rather than slide, only 71.4% of the gravity vector goes into accelerating the ball forward. The remaining 28.6% of the force is strictly used to overcome the ball's rotational inertia and make it spin.

The normal force on the ball has a coriolois and centripetal term
$$
N = mg \cos \theta + m x \ddot{\theta} + 2m\dot{x}\dot{\theta}
$$
which leads to the condition
$$
g \cos \theta + x \ddot{\theta} + 2\dot{x}\dot{\theta} \gt 0
$$
for the ball to be in contact with the beam.

Note that we assume a perfect (infintely stiff) actuator for the beam angle $\theta$ which is unaffected by the ball's mass or position.

# Implementation in bdsim

## Simulation

We start with the standard imports, construct a simulator object, and use that to construct
a block diagram with all the `bdsim` blocks included as methods.

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True, animation_rate=10)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

Next up, we define the parameters of the system

In [ ]:
m = 0.1  # mass of ball
R = 0.0145  # radius of ball
l = 0.5  # half length of beam
g = 9.81  # gravity
b = 0.05  # ball friction

The system can be modeled as a chain of two integrators.  The first integrates ball acceleration $\ddot{x}$ to velocity $\dot{x}$ to position $x$.  The initial position and velocity is zero.

In [ ]:
x_dot = bd.INTEGRATOR(name="x_dot")
x = bd.INTEGRATOR(name="x", inputs=[x_dot])

Note that we use the `inputs` parameter of the second integrator to the output of the first integrator.

The system is initially in an unstable equilbrium with all states zero, the beam horizontal and the ball stationary at the centre.  We apply brief tilt to the beam to get the ball rolling. ($t \in [1, 1.2]$).

$\theta$ is set by the controller, but the equations of motion require its derivatives $\dot{\theta}$ and $\ddot{\theta}$.  We compute these by feeding $\theta$ into a cascade of differentiators.  A pure differentiator with a transfer function of $s$ is not implementable -- it is noncausal -- so  the differentiator includes a first-order denominator with a pole at $\alpha$.  In this case we choose a time constant of 0.1s which correponds to a pole at $2\pi/T$

In [ ]:
K = np.zeros((1,2))

theta = bd.GAIN(1)
theta[0] = bd.PIECEWISE((0,0), (1, 0.1), (1.2, 0), name="pulse") #- (bd.STATE(["x_dot", "x"]) >> bd.GAIN(K, premul=True,name="K"))

theta_dot = theta >> bd.DERIV(alpha=10, name="theta_dot")
theta_ddot = theta_dot >>bd.DERIV(alpha=10, name="theta_ddot")

We use a pair of function blocks to compute the sine and cosine of the beam angle

In [ ]:
sin_theta = theta >> bd.FUNCTION(math.sin, name="sin(theta)")
cos_theta = theta >> bd.FUNCTION(math.cos, name="cos(theta)")

Next we compute the normal force of the ball on the beam, it should be positive while the ball is supported by the beam.  If negative it means that the beam has accelerated downward from the ball leaving it in freefall.  This would be the result of a very aggresive controller.

In [ ]:
N =  m*g*cos_theta + m*x*theta_ddot + 2*m*x_dot*theta_dot

In [ ]:
x_dot[0] = (m*x*theta_dot**2 - m*g*sin_theta - b*x_dot) / (5.0/7.0)

In [ ]:
# display the system state evolution on a scope and animation
scope = bd.SCOPE(
    nin=2,
    inputs=[x, theta],
    labels=("x", r"$\theta$"),
    loc="lower right",
)


To include a graphical animation of our cart and pole system we will use simple Matplotlib 
patch graphics.  This kind of code is tedious to write, and it can be outsourced to a coding agent.

In [ ]:
import matplotlib.patches as patches
import matplotlib.transforms as transforms


def ballbeam_init(self, fig, ax, b=l, R=R):
    # b is the half length of the beam and R the radius of the ball, bound when this
    # is defined since R is reused later for the LQR design.  The beam is 2b wide and
    # R tall, centered on the pivot at the origin; the ball sits on top of it.
    lim = 1.25 * b  # the beam stays inside this circle at any angle
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(True, linestyle="--")

    # Everything is drawn in the beam frame, with the beam level.  The update
    # function rotates and slides it.
    beam = patches.Rectangle(
        (-b, -R / 2), 2 * b, R, facecolor="0.5", edgecolor="none"
    )
    ball = patches.Circle((0, R / 2 + R), R, facecolor="red", edgecolor="none")
    pivot = patches.Circle((0, 0), R / 3, facecolor="black", edgecolor="none", zorder=3)

    for patch in (beam, ball, pivot):
        ax.add_patch(patch)

    self.beam = beam
    self.ball = ball
    self.exaggeration = 1.0  # angle exaggeration, change with anim.exaggeration = 3

and the actual update

In [ ]:
def ballbeam_update(self, t, u):
    x, theta = u  # ball position along the beam, beam angle (rad)
    angle = self.exaggeration * theta  # increasing angle raises the right end

    # rotate about the pivot, which is the origin
    rotate = transforms.Affine2D().rotate(angle) + self.ax.transData
    self.beam.set_transform(rotate)

    # slide the ball along the beam, then rotate it with the beam
    slide = transforms.Affine2D().translate(x, 0).rotate(angle) + self.ax.transData
    self.ball.set_transform(slide)

These two functions are passed to an `ANIMATION` block driven by the outputs of the `x` and `theta` blocks.  These inputs are the elements of `u` passed to `cartpole_update`.

In [ ]:
anim = bd.ANIMATION(
    init=ballbeam_init, update=ballbeam_update, nin=2, inputs=[x, theta]
)

Our system is built, we will compile it, and display a summary

In [ ]:
bd.compile()  # check the diagram
print(bd)

Which has 47 blocks, although we have only explicitly created 4 integrators, 2 functions, 2 steps, an animation and a scope.  The majority of the blocks were created by the arithmetic expressions for the equations of motion and system energy.  

Here is a full list of blocks.

In [ ]:
bd.report_summary()

where the implicitly created blocks have names prefixed by `_`.

To create all these blocks and connect them would be both tedious and error prone.  The power of `bdsim` is that it lets you move effortlessly between expression in terms of blocks or using Python expressions.

In [ ]:
out = sim.run(bd, T=10, watch=[N, x, theta])  # simulate for 5s
print(out)


## Linearization

In [ ]:
X = bd.linearize(t= 0, inputs=["pulse"], full=True)
print(X)



# Control 

## Control design

Now we can design a linear controller using the Python Control package

```
$ pip install control
```

In [ ]:
import control as ct

# Create the state-space system (optional, but good practice)
sys = ct.ss(X.A[:2,:2], X.B[:2,:], X.C[:,:2], X.D) #, states=X.xindex.names[:2], inputs=X.uindex.names, outputs=X.yindex.names)

# 2. Define the cost matrices
# Q must be positive semi-definite (penalizes state errors)
# R must be positive definite (penalizes control effort)
Q = np.diag([0.2, 10])  # weights for [x_dot, x]
R = np.array([[10]]) # weight for control input (force F)

# 3. Compute the LQR optimal gain matrix K
# control.lqr returns:
# K: The optimal feedback gain matrix (u = -Kx)
# S: The solution to the algebraic Riccati equation
# E: The eigenvalues (poles) of the closed-loop system (A - BK)
K, S, E = ct.lqr(sys, Q, R)

# Note: You can also pass the arrays directly without creating a sys object:
# K, S, E = ct.lqr(A, B, Q, R)

print("Optimal Feedback Gain (K): \n", K)
print("Closed-loop Poles (E): \n", E)

In [ ]:
K.shape

## Closed-loop simulation

For closed-loop simulation we can reuse most of the model that we've already built. We'll do some surgery on the model:

* remove the existing pulse generator we used to verify the model
* add a square wave demand for the ball position
* convert the LQ regulator to a controller

In [ ]:
# bd["K"].K = K  # set the gain block to the computed LQR gain
bd.delete_block("pulse")  # remove the old theta block
x = bd.STATE(["x_dot", "x"], name="state2") >> bd.DEMUX(2)
setpoint = bd.WAVEFORM("square", amplitude=0.3, freq=0.25, name="setpoint")  # create a triangle waveform for the setpoint
u = x[0] * (-K[0, 0]) + (setpoint - x[1]) * (-K[0, 1])
theta[0] = u  # set the control input to the computed LQR control law


# theta = bd.PIECEWISE((0,0), (1, 0.1), (1.2, 0), name="theta") - (bd.STATE(["x_dot", "x"]) >> bd.GAIN(K, premul=True,name="K"))

# 

In [ ]:
print(bd["state2"])
print(x)

In [ ]:
bd.compile()
out = sim.run(bd, T=5, watch=[N, x, theta])  # simulate for 5s